# DataDojo — Перевод EN→RU с Gemma 4 + QLoRA

**Задача:** Дообучить большую языковую модель для перевода с английского на русский  
**Базовая модель:** `google/gemma-4-e2b-it` (2 млрд параметров, инструктивная)  
**Метод:** QLoRA — 4-битное квантование + низкоранговая адаптация (Low-Rank Adaptation)  
**Датасет:** Helsinki-NLP/opus-100 (en-ru), 20 000 примеров  
**Результат:** LoRA-адаптер обучен за 1 эпоху, train_loss = 1.413

---

## Почему QLoRA?

У Gemma 4 2B ~5.1 млрд параметров — это слишком много, чтобы полностью дообучить на одной GPU.

**QLoRA** решает это в два шага:
1. **4-битное квантование (BitsAndBytes)** — загружает базовую модель в 4-битной точности, снижая VRAM с ~10 ГБ до ~3 ГБ
2. **LoRA** — вместо обновления всех весов обучает маленькие низкоранговые адаптеры (~38 млн обучаемых параметров из 5.1 млрд = 0.74%)

| | Полное дообучение | QLoRA |
|---|---|---|
| Обучаемых параметров | 5.1 млрд | 38 млн (0.74%) |
| Нужно VRAM | ~40 ГБ | ~6 ГБ |
| Скорость | Медленно | Быстро |

---

## Структура проекта

| Файл / папка | Описание |
|---|---|
| `datadojo.ipynb` | Этот ноутбук — пошаговый разбор обучения и инференса |
| `Yandex/gemma_solution/solution.py` | Контестный инференс-скрипт (Docker, читает `input.pickle` → пишет `output.json`) |
| `Yandex/gemma_solution/train_lora.py` | Скрипт обучения QLoRA-адаптера (CLI, поддержка OPUS-100 и JSONL) |
| `Yandex/gemma_solution/merge_lora.py` | Слияние LoRA-адаптера с базовой моделью |
| `Yandex/gemma_solution/Dockerfile` | Docker-образ для submission |
| `Yandex/gemma_solution/LORA_TRAINING.md` | Инструкция по запуску обучения на Kaggle |
| `Yandex/gemma_solution/weights/` | Веса модели (~9.5 ГБ, в .gitignore) |

## Шаг 1 — Авторизация и загрузка модели

In [ ]:
from huggingface_hub import login
import os

# Перед запуском задайте переменную окружения HF_TOKEN
login(os.getenv("HF_TOKEN"))

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "google/gemma-4-e2b-it"
use_cuda = torch.cuda.is_available()

tokenizer = AutoTokenizer.from_pretrained(model_id)

if use_cuda:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
    )
else:
    print("GPU не найдена, загружаю на CPU (медленно — для обучения используйте Kaggle/Colab)")
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map="cpu",
        torch_dtype=torch.float32,
    )

model.eval()
print(f"Модель загружена на {'GPU' if use_cuda else 'CPU'}.")

## Шаг 2 — Инференс (базовая модель, до дообучения)

In [ ]:
def translate(text, max_new_tokens=1200):
    messages = [{
        "role": "user",
        "content": (
            "Переведи следующий английский абзац на русский язык.\n"
            "Ответ должен быть только на русском языке, без пояснений и комментариев.\n"
            "Сохрани имена, даты, числа, технические термины и порядок предложений.\n\n"
            f"Английский абзац:\n{text}\n\n"
            "Русский перевод:"
        ),
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][input_len:]
    result = tokenizer.decode(generated, skip_special_tokens=True).strip()

    for prefix in ["Russian:", "Translation:", "Russian translation:", "Перевод:", "Русский перевод:"]:
        if result.startswith(prefix):
            result = result[len(prefix):].strip()

    return result

In [ ]:
src = """In the interview, he said that the company had been working on the project for more than two years.
The team wanted to make the tool useful not only for engineers, but also for writers, designers, and teachers.
According to him, the most difficult part was not building the first prototype,
but making sure that the product remained reliable when thousands of people used it at the same time."""

print(translate(src))

## Шаг 3 — Дообучение с QLoRA

Обучение выполняется через `train_lora.py` — самодостаточный скрипт, который делает:
- Загрузку датасета (OPUS-100 en-ru, 20 тыс. примеров)
- Построение промптов по chat-шаблону с маскированием меток (обучаются только токены перевода)
- Настройку QLoRA через PEFT
- HuggingFace Trainer с gradient checkpointing

**Конфигурация LoRA:**
| Параметр | Значение | Смысл |
|---|---|---|
| `r` | 16 | Ранг матриц адаптера |
| `lora_alpha` | 32 | Коэффициент масштабирования |
| `lora_dropout` | 0.05 | Регуляризация |
| `target_modules` | all-linear | LoRA применяется ко всем линейным слоям |

**Конфигурация обучения:**
| Параметр | Значение |
|---|---|
| Датасет | Helsinki-NLP/opus-100 (en-ru), 20 тыс. примеров |
| Эпохи | 1 |
| Размер батча | 1 + grad_accum=16 (эффективный батч = 16) |
| Learning rate | 2e-4 с косинусным планировщиком |
| Макс. длина последовательности | 512 токенов |
| Оптимизатор | paged_adamw_8bit |

In [ ]:
# Запуск дообучения
# При необходимости поправьте пути под своё окружение

!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python train_lora.py \
  --model_name_or_path google/gemma-4-e2b-it \
  --dataset_name Helsinki-NLP/opus-100 \
  --dataset_config en-ru \
  --dataset_split train \
  --max_samples 20000 \
  --output_dir ./enru-lora \
  --epochs 1 \
  --batch_size 1 \
  --grad_accum 16 \
  --max_seq_length 512 \
  --val_fraction 0 \
  --save_steps 25 \
  --resume auto

## Результаты обучения

| Метрика | Значение |
|---|---|
| Обучаемых параметров | 37 920 768 (0.74% от всех) |
| Train loss (финальный) | **1.413** |
| Время обучения | ~3 ч 20 мин |
| Примеров в секунду | 1.664 |
| Шагов | 625 |

## Шаг 4 — Инференс с дообученным адаптером

In [ ]:
from peft import PeftModel

ADAPTER_PATH = "./enru-lora"

# Загружаем базовую модель + подключаем обученный LoRA-адаптер
ft_model = PeftModel.from_pretrained(model, ADAPTER_PATH)
ft_model.eval()
print("Дообученная модель загружена.")

In [ ]:
def translate_ft(text, max_new_tokens=1200):
    messages = [{
        "role": "user",
        "content": (
            "Переведи следующий английский абзац на русский язык.\n"
            "Ответ должен быть только на русском языке, без пояснений и комментариев.\n"
            "Сохрани имена, даты, числа, технические термины и порядок предложений.\n\n"
            f"Английский абзац:\n{text}\n\n"
            "Русский перевод:"
        ),
    }]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to(ft_model.device)

    input_len = inputs["input_ids"].shape[1]

    with torch.no_grad():
        output = ft_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id,
        )

    generated = output[0][input_len:]
    result = tokenizer.decode(generated, skip_special_tokens=True).strip()

    for prefix in ["Russian:", "Translation:", "Russian translation:", "Перевод:", "Русский перевод:"]:
        if result.startswith(prefix):
            result = result[len(prefix):].strip()

    return result


# Тест на том же предложении, что и до дообучения
print("=== Дообученная модель ===")
print(translate_ft(src))

## Итог

- Загрузили **Gemma 4 2B** в 4-битном квантовании (~3 ГБ VRAM вместо ~10 ГБ)
- Применили **LoRA**-адаптеры — обучили лишь 0.74% всех параметров
- Дообучили на **20 000 парах EN→RU** из OPUS-100 за 1 эпоху
- Финальный train loss: **1.413** (сходится, без переобучения — loss падал на всём протяжении)
- Адаптер сохранён в `enru-lora/` — ~150 МБ против ~5 ГБ для полных весов

**Как улучшить дальше:**
- Обучать 2-3 эпохи
- Использовать больше данных (в opus-100 1 млн пар, использовано только 20 тыс.)
- Оценивать метриками BLEU или chrF на отложенной выборке
- Попробовать больший ранг LoRA (r=32 или r=64)